In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(
    "../data/processed/customers_eda.csv"
)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TenureGroup,SeniorCitizenLabel
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0-12 Months,Non-Senior
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,One year,No,Mailed check,56.95,1889.50,No,25-48 Months,Non-Senior
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,0-12 Months,Non-Senior
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,25-48 Months,Non-Senior
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,0-12 Months,Non-Senior


In [3]:
df.columns.tolist()

['customerID',
 'gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges',
 'Churn',
 'TenureGroup',
 'SeniorCitizenLabel']

Covert the text target to numerical target ["No" -> 0, "Yes" -> 1]

In [4]:
df["ChurnFlag"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

In [8]:
df[["Churn", "ChurnFlag"]].head(10)

,Churn,ChurnFlag
0,No,0
1,No,0
2,Yes,1
3,No,0
4,Yes,1
5,Yes,1
6,No,0
7,No,0
8,Yes,1
9,No,0


In [9]:
df["ChurnFlag"].isnull().sum()

np.int64(0)

Create a monthly charge category

Instead of only having the numerical

In [10]:
df["MonthlyCharges"].describe()

count    7043.000000
mean       64.761692
std        30.090047
min        18.250000
25%        35.500000
50%        70.350000
75%        89.850000
max       118.750000
Name: MonthlyCharges, dtype: float64

In [11]:
low_threshold = df["MonthlyCharges"].quantile(0.33)
high_threshold = df["MonthlyCharges"].quantile(0.66)

print(low_threshold, high_threshold)

50.2 83.2


In [12]:
df["MonthlyChargeCategory"] = pd.cut(
    df["MonthlyCharges"],
    bins=[
        -np.inf,
        low_threshold,
        high_threshold,
        np.inf
    ],
    labels=[
        "Low",
        "Medium",
        "High"
    ]
)

In [13]:
df["MonthlyChargeCategory"].value_counts()

MonthlyChargeCategory
High      2392
Low       2327
Medium    2324
Name: count, dtype: int64

We can calculate approximately how many services a customer uses

In [14]:
service_columns = [
    "PhoneService",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

for col in service_columns:
    print(col)
    print(df[col].value_counts())
    print()

PhoneService
PhoneService
Yes    6361
No      682
Name: count, dtype: int64

MultipleLines
MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64

OnlineSecurity
OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64

OnlineBackup
OnlineBackup
No                     3088
Yes                    2429
No internet service    1526
Name: count, dtype: int64

DeviceProtection
DeviceProtection
No                     3095
Yes                    2422
No internet service    1526
Name: count, dtype: int64

TechSupport
TechSupport
No                     3473
Yes                    2044
No internet service    1526
Name: count, dtype: int64

StreamingTV
StreamingTV
No                     2810
Yes                    2707
No internet service    1526
Name: count, dtype: int64

StreamingMovies
StreamingMovies
No                     2785
Yes                    2732
No internet 

In [15]:
df["ServiceCount"] = (
    df[service_columns]
    .eq("Yes")
    .sum(axis=1)
)

In [16]:
df["ServiceCount"].describe()

count    7043.000000
mean        3.362914
std         2.062031
min         0.000000
25%         1.000000
50%         3.000000
75%         5.000000
max         8.000000
Name: ServiceCount, dtype: float64

In [17]:
df[["customerID", "ServiceCount"]].head()

,customerID,ServiceCount
0,7590-VHVEG,1
1,5575-GNVDE,3
2,3668-QPYBK,3
3,7795-CFOCW,3
4,9237-HQITU,1


Create a high-value customer flag

We'll define a high-value customer as someone whose monthly charge is above the median.

In [18]:
monthly_median = df["MonthlyCharges"].median()

df["HighValueCustomer"] = np.where(
    df["MonthlyCharges"] >= monthly_median,
    "Yes",
    "No"
)

In [19]:
df["HighValueCustomer"].value_counts()

HighValueCustomer
Yes    3524
No     3519
Name: count, dtype: int64

Improve the tenure feature

In [20]:
df["TenureGroup"].value_counts()

TenureGroup
49-72 Months    2239
0-12 Months     2186
25-48 Months    1594
13-24 Months    1024
Name: count, dtype: int64

In [22]:
df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=[
        "0-12 Months",
        "13-24 Months",
        "25-48 Months",
        "49-72 Months"
    ]
)

Create average monthly revenue

In [24]:
df["EstimatedAnnualCharges"] = (
    df["MonthlyCharges"] * 12
)

Check the new features

In [25]:
new_features = [
    "ChurnFlag",
    "MonthlyChargeCategory",
    "ServiceCount",
    "HighValueCustomer",
    "TenureGroup",
    "EstimatedAnnualCharges"
]

df[new_features].head(10)

,ChurnFlag,MonthlyChargeCategory,ServiceCount,HighValueCustomer,TenureGroup,EstimatedAnnualCharges
0,0,Low,1,No,0-12 Months,358.2
1,0,Medium,3,No,25-48 Months,683.4
2,1,Medium,3,No,0-12 Months,646.2
3,0,Low,3,No,25-48 Months,507.6
4,1,Medium,1,Yes,0-12 Months,848.4
5,1,High,5,Yes,0-12 Months,1195.8
6,0,High,4,Yes,13-24 Months,1069.2
7,0,Low,1,No,0-12 Months,357.0
8,1,High,6,Yes,25-48 Months,1257.6
9,0,Medium,3,No,49-72 Months,673.8


Check coorelation

In [26]:
numeric_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "ServiceCount",
    "EstimatedAnnualCharges",
    "ChurnFlag"
]

df[numeric_features].corr()["ChurnFlag"].sort_values(
    ascending=False
)

ChurnFlag                 1.000000
MonthlyCharges            0.193356
EstimatedAnnualCharges    0.193356
SeniorCitizen             0.150889
ServiceCount             -0.067264
TotalCharges             -0.199037
tenure                   -0.352229
Name: ChurnFlag, dtype: float64

Select features for ML

In [27]:
features = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "tenure",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "MonthlyCharges",
    "TotalCharges",
    "ServiceCount"
]

In [28]:
target = "ChurnFlag"

In [29]:
X = df[features]
y = df[target]

In [30]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 20)
y shape: (7043,)


Train test split

In [31]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [32]:
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("Training churn rate:")
print(y_train.mean())

print("Testing churn rate:")
print(y_test.mean())

Training rows: 5634
Testing rows: 1409
Training churn rate:
0.2653532126375577
Testing churn rate:
0.2654364797728886


Build the preprocessing pipeline

Machine learning models cannot directly understand:

Contract = Month-to-month

We need to convert categorical variables into numerical representations.

In [34]:
categorical_features = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

numeric_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "ServiceCount"
]

In [35]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numeric_features
        )
    ]
)

In [36]:
df.to_csv(
    "../data/processed/customers_features.csv",
    index=False
)

We'll eventually create:

Raw Features
     ↓
One-Hot Encoding
     ↓
Logistic Regression

Instead of manually transforming the data.

This is cleaner and helps prevent data leakage.